# PY 03 — PyTorch Basics

PyTorch is the library that powers most modern AI research and production systems — including the models behind ChatGPT.

By the end you'll understand:
- **Tensors** — PyTorch's version of NumPy arrays
- **Autograd** — how PyTorch computes gradients automatically
- **A simple model** — build and train a linear model from scratch
- **The training loop** — forward → loss → backward → update

This is the engine running under every AI phase that follows.

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {torch.backends.mps.is_available()}")

---
## 1. Tensors — PyTorch's arrays

A tensor is just a multi-dimensional array — exactly like NumPy, but with two superpowers:
1. Can run on GPU
2. Tracks gradients for automatic differentiation

In [ ]:
# Creating tensors — same feel as NumPy
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.zeros(4)
c = torch.ones(3, 4)
d = torch.randn(2, 3)   # standard normal

print("a:", a)
print("b:", b)
print("c shape:", c.shape)
print("d:\n", d)

In [ ]:
# Tensor ↔ NumPy conversion
np_array = np.array([1.0, 2.0, 3.0])
tensor   = torch.from_numpy(np_array)   # NumPy → Tensor
back     = tensor.numpy()               # Tensor → NumPy

print("NumPy:  ", np_array)
print("Tensor: ", tensor)
print("Back:   ", back)

# Same operations as NumPy
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])
print("\nx + y:", x + y)
print("x @ y:", x @ y)   # dot product

---
## 2. Autograd — automatic differentiation

This is PyTorch's killer feature. When you compute with tensors that have `requires_grad=True`, PyTorch silently tracks every operation. When you call `.backward()`, it computes gradients automatically.

This is how neural networks learn — gradients tell each weight which direction to move to reduce the loss.

In [ ]:
# Simple example: y = x^2, dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print(f"x = {x}")
print(f"y = x² = {y}")

y.backward()  # compute gradients
print(f"dy/dx = {x.grad}  (expected: 2 × 3 = 6.0)")

In [ ]:
# More realistic: loss = mean squared error
# prediction = w * x + b

w = torch.tensor(2.0, requires_grad=True)  # weight
b = torch.tensor(0.0, requires_grad=True)  # bias
x = torch.tensor(4.0)                      # input
y_true = torch.tensor(10.0)                # target

y_pred = w * x + b                         # forward pass
loss   = (y_pred - y_true) ** 2            # MSE loss

print(f"Prediction: {y_pred.item():.2f}")
print(f"Loss:       {loss.item():.2f}")

loss.backward()                            # backward pass
print(f"\ndLoss/dw = {w.grad.item():.2f}")
print(f"dLoss/db = {b.grad.item():.2f}")
print("\nThese gradients tell us how to adjust w and b to reduce the loss.")

---
## 3. Building a model with nn.Module

Real PyTorch models are classes that inherit from `nn.Module`. This gives you parameter tracking, GPU support, and save/load for free.

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        return self.linear(x)

model = LinearModel(input_dim=4, output_dim=1)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")
# 4 weights + 1 bias = 5 parameters

# Forward pass
sample_input = torch.randn(1, 4)  # batch of 1, 4 features
output = model(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {output.shape}")

---
## 4. The training loop

Every neural network training follows this exact pattern:

```
for each batch:
    1. Forward pass  — compute predictions
    2. Compute loss  — how wrong are we?
    3. Backward pass — compute gradients
    4. Update weights— move in the right direction
    5. Zero gradients— reset for next batch
```

Let's train a model to learn the function `y = 3x + 2`.

In [ ]:
# Generate training data: y = 3x + 2 + noise
torch.manual_seed(42)
X = torch.linspace(-1, 1, 100).unsqueeze(1)  # shape (100, 1)
y = 3 * X + 2 + torch.randn_like(X) * 0.2    # true relationship + noise

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

plt.figure(figsize=(7, 4))
plt.scatter(X.numpy(), y.numpy(), alpha=0.4, s=20, color='steelblue')
plt.xlabel('X'); plt.ylabel('y')
plt.title('Training Data: y = 3x + 2 + noise')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Model, loss function, optimiser
model     = nn.Linear(1, 1)               # one input, one output
criterion = nn.MSELoss()                  # mean squared error
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []

# Training loop
for epoch in range(100):
    # 1. Forward pass
    y_pred = model(X)
    
    # 2. Compute loss
    loss = criterion(y_pred, y)
    losses.append(loss.item())
    
    # 3. Backward pass
    loss.backward()
    
    # 4. Update weights
    optimizer.step()
    
    # 5. Zero gradients
    optimizer.zero_grad()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# What did the model learn?
w = model.weight.item()
b = model.bias.item()
print(f"\nLearned: y = {w:.2f}x + {b:.2f}")
print(f"True:    y = 3.00x + 2.00")

In [ ]:
# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(losses, color='steelblue', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Fit
with torch.no_grad():
    y_fit = model(X).numpy()
ax2.scatter(X.numpy(), y.numpy(), alpha=0.4, s=20, color='steelblue', label='Data')
ax2.plot(X.numpy(), y_fit, color='tomato', linewidth=2, label='Model fit')
ax2.set_xlabel('X'); ax2.set_ylabel('y')
ax2.set_title('Model Fit')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. A deeper model

Real AI models stack many layers. Here's a small multi-layer network — the same pattern used in GPT, BERT, and every Transformer.

In [ ]:
class MLP(nn.Module):
    """A simple multi-layer perceptron."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

mlp = MLP()
print(mlp)

total = sum(p.numel() for p in mlp.parameters())
print(f"\nTotal parameters: {total:,}")

# Forward pass
x = torch.randn(8, 4)   # batch of 8 examples, 4 features each
out = mlp(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

---
## ✓ What you learned

- **Tensors** are PyTorch's arrays — same feel as NumPy but GPU-ready
- **Autograd** tracks operations and computes gradients automatically
- **nn.Module** is the base class for all PyTorch models
- **The training loop** is always: forward → loss → backward → step → zero_grad
- **nn.Sequential** stacks layers cleanly

You've now trained your first neural network. Every AI model in this course uses this exact same loop — just with more layers and fancier architectures.

Up next → **PY 04: Before You Start AI** — tools, glossary, and key papers.